# 部署侧的算子融合：QDQ 怎么被吸收掉，加速从哪来

对应文章：《扩展篇 E5：伪量化的另一半——部署时 Q/DQ 算子怎么被吸收，加速从哪来》

第 17 篇把「训练层的 FQ / 表示层的 QDQ / 执行层的整数 kernel」分成了三层，但执行层只写到了
「引擎会融合」为止——**融合的代数、多出来的修正项、加速到底来自哪几笔账都没展开**。这一本把它们钉死。

核心要验证的五件事：

| 组 | 验证什么 |
|---|---|
| A | 融合恒等式：浮点 QDQ 参考 == 整数 GEMM + 三项修正（逐位） |
| B | 三项修正的消融：漏掉任何一项，误差是什么量级 |
| C | 对称量化为什么工程上更省：两项修正自动归零 |
| D | requantize 的定点乘子：位宽够不够、舍入偏置从哪来 |
| E | 三笔收益账：算力 / 带宽 / 显存，谁才是 W4A16 的主收益 |

所有任务均为合成探针（synthetic probe），不是真实模型精度。
解释器：`/Users/congyuan/Software/miniconda3/envs/torch/bin/python`（torch 2.10.0 / numpy 1.26.4，CPU）。

In [1]:
import os, sys, json, platform, warnings
warnings.filterwarnings("ignore")
import numpy as np
import torch

MODE = "full"          # "smoke" 只跑 A/B/C 的小规模，"full" 跑全部
SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

N_ASSERT = 0
def check(cond, msg):
    global N_ASSERT
    assert cond, f"ASSERT FAILED: {msg}"
    N_ASSERT += 1
    print(f"  [ok] {msg}")

print(f"python {platform.python_version()} | torch {torch.__version__} | numpy {np.__version__} | machine {platform.machine()}")
print(f"MODE = {MODE} | SEED = {SEED}")

python 3.11.14 | torch 2.10.0 | numpy 2.2.6 | machine arm64
MODE = full | SEED = 0


## 0. 执行层的前提：本机到底有没有整数 kernel

在推导融合代数之前，先确认一件事：**QDQ 图导出成功不等于整数计算会发生**。
第 17 篇 §7.3 说「没有整数 kernel 的边界会被反量化回浮点」，这里把它变成可复现的实测。

In [2]:
import torch.nn as nn
import torch.ao.quantization as tqao

print("supported_engines:", torch.backends.quantized.supported_engines)
print()

# (1) 底层量化张量原语是否可用 —— 融合代数只需要它
wq = torch.quantize_per_tensor(torch.randn(4, 8), 0.05, 0, torch.qint8)
print("(1) quantize_per_tensor 可用:", wq.dtype, "| int_repr:", wq.int_repr().dtype)
wqc = torch.quantize_per_channel(
    torch.randn(4, 8), torch.tensor([0.05]*4), torch.zeros(4, dtype=torch.int64), 0, torch.qint8)
print("    quantize_per_channel 可用:", wqc.dtype)

# (2) 裸 Linear 走 quantize_dynamic：静默返回浮点模型
lin = nn.Linear(64, 32)
qd = tqao.quantize_dynamic(lin, {nn.Linear}, dtype=torch.qint8)
bare_is_float = (type(qd) is nn.Linear) and (not qd.weight.is_quantized)
print(f"\n(2) 裸 Linear -> type={type(qd).__name__} is_nn.Linear={type(qd) is nn.Linear} "
      f"weight.dtype={qd.weight.dtype} is_quantized={qd.weight.is_quantized}")
print(f"    >>> 静默保持浮点（无异常、无警告）: {bare_is_float}")

# (3) 嵌套在 Sequential 里的 Linear：直接抛异常
raised = None
try:
    tqao.quantize_dynamic(nn.Sequential(nn.Linear(8, 4)), {nn.Linear}, dtype=torch.qint8)
except Exception as e:
    raised = f"{type(e).__name__}: {str(e)[:70]}"
print(f"\n(3) Sequential(nn.Linear) -> {raised}")

# (4) eager 静态量化：prepare 只插了激活 observer，权重 observer 缺失
lin2 = nn.Linear(64, 32)
lin2.qconfig = tqao.get_default_qconfig("qnnpack")
prep = tqao.prepare(lin2)          # 注意：prepare 默认返回 deepcopy
print(f"\n(4) prepare 返回 type={type(prep).__name__}")
print(f"    has activation_post_process = {hasattr(prep, 'activation_post_process')}")
print(f"    has weight_fake_quant      = {hasattr(prep, 'weight_fake_quant')}")
cv = tqao.convert(prep)
print(f"    convert 后 type={type(cv).__name__} | has _packed_params = {hasattr(cv, '_packed_params')}")

check(bare_is_float, "裸 Linear 的 quantize_dynamic 静默返回浮点模型（无异常）")
check(raised is not None and "NoQEngine" in raised, "嵌套 Linear 抛 NoQEngine，整数 kernel 确实缺失")
check(hasattr(prep, "activation_post_process") and not hasattr(prep, "weight_fake_quant"),
      "eager prepare 只插入激活 observer，权重 observer 缺失")
check(not hasattr(cv, "_packed_params"), "convert 后无 _packed_params，即未产出量化模块")

print("\n>>> 结论：本机（ARM macOS）无量化 engine。"
      "QDQ 图可以导出，但整数 kernel 缺席时执行层会退回浮点——"
      "这正是第 17 篇『导出成 QDQ ≠ 部署会变快』的机器级证据。")

supported_engines: ['qnnpack']

(1) quantize_per_tensor 可用: torch.qint8 | int_repr: torch.int8
    quantize_per_channel 可用: torch.qint8

(2) 裸 Linear -> type=Linear is_nn.Linear=True weight.dtype=torch.float32 is_quantized=False
    >>> 静默保持浮点（无异常、无警告）: True

(3) Sequential(nn.Linear) -> RuntimeError: Didn't find engine for operation quantized::linear_prepack NoQEngine

(4) prepare 返回 type=Linear
    has activation_post_process = True
    has weight_fake_quant      = False
    convert 后 type=Linear | has _packed_params = False
  [ok] 裸 Linear 的 quantize_dynamic 静默返回浮点模型（无异常）
  [ok] 嵌套 Linear 抛 NoQEngine，整数 kernel 确实缺失
  [ok] eager prepare 只插入激活 observer，权重 observer 缺失
  [ok] convert 后无 _packed_params，即未产出量化模块

>>> 结论：本机（ARM macOS）无量化 engine。QDQ 图可以导出，但整数 kernel 缺席时执行层会退回浮点——这正是第 17 篇『导出成 QDQ ≠ 部署会变快』的机器级证据。


## A. 融合恒等式：DQ 怎么被吸收进 GEMM

设权重与激活各自仿射量化：

$$W \approx s_W(q_W - z_W),\qquad x \approx s_x(q_x - z_x)$$

把它代进矩阵乘（$$K$$ 是 contraction 维）：

$$s_W(q_W - z_W)\;\cdot\;s_x(q_x - z_x)
  = s_W s_x\Big[\underbrace{q_W q_x}_{\text{整数 GEMM}}
      - z_x\underbrace{\textstyle\sum_k q_W[:,k]}_{\text{行和}}
      - z_W\underbrace{\textstyle\sum_k q_x[k]}_{\text{标量}}
      + K\,z_W z_x\Big]$$

**这就是融合的全部内容**：四个量里只有第一个是真 GEMM，后三个都是可以在部署前预计算、
或在 epilogue 里顺手加上的修正项。所谓「把 QDQ 吸收掉」，指的就是这三项修正被并进 kernel，
而不再各写一次反量化。

In [3]:
def qparams(x, bits=8, signed=True, symmetric=False):
    qmin, qmax = (-(1 << (bits - 1)), (1 << (bits - 1)) - 1) if signed else (0, (1 << bits) - 1)
    x = np.asarray(x, dtype=np.float64)
    if symmetric:
        s = float(np.abs(x).max()) / qmax; zp = 0
    else:
        mn, mx = float(x.min()), float(x.max())
        s = (mx - mn) / (qmax - qmin)
        zp = int(np.clip(qmin - np.round(mn / s), qmin, qmax))
    return (s if s > 0 else 1.0), zp, qmin, qmax

def quantize(x, s, zp, qmin, qmax):
    return np.clip(np.round(np.asarray(x) / s) + zp, qmin, qmax).astype(np.int64)

def dequantize(q, s, zp):
    return s * (np.asarray(q) - zp)

def rshift(v, sh):
    """带四舍五入的右移（naive 口径，保留 D4 要演示的偏置）。

    注意这里刻意写成「加半个 LSB 再算术右移」——它是工程上最常见的写法，
    但在 D4 会被证明对负数引入 -0.5 的系统性偏置。
    """
    return (v + (1 << (sh - 1))) >> sh

def rshift_sym(v, sh):
    """对称四舍五入右移（向量化）：正负两侧都取 round-half-up。

    不能写成 `divmod` 后对负数补 -1 —— Python 的 `//` 是 floor 语义，
    那样会让负数分支整体少 1（实测输出码字恰好全部偏 -2，见 D4）。
    """
    v = np.asarray(v, dtype=object)
    half = 1 << (sh - 1)
    sign = np.where(v >= 0, 1, -1)
    return sign * ((sign * v + half) >> sh)

def rshift_sym_scalar(v, sh):
    """标量版，用于 D4 的逐样本偏置统计。"""
    s = -1 if v < 0 else 1
    return s * ((s * v + (1 << (sh - 1))) >> sh)

OUT, K = (128, 256) if MODE == "full" else (32, 64)
W = np.random.randn(OUT, K) * 0.1
b = np.random.randn(OUT) * 0.01
x = np.random.randn(K)

s_w, z_w, wmin, wmax = qparams(W, 8, signed=True)
s_x, z_x, xmin, xmax = qparams(x, 8, signed=False)
print(f"s_W={s_w:.6f} z_W={z_w} | s_x={s_x:.6f} z_x={z_x} | K={K}")

W_q = quantize(W, s_w, z_w, wmin, wmax)
x_q = quantize(x, s_x, z_x, xmin, xmax)

# 参考：显式反量化后再做浮点 GEMM（这就是 QDQ 图的语义）
y_ref = dequantize(W_q, s_w, z_w) @ dequantize(x_q, s_x, z_x) + b

# 融合：整数 GEMM + 三项修正
acc      = W_q @ x_q
rowsum_W = W_q.sum(axis=1)
sum_x    = x_q.sum()
int_expr = acc - z_x * rowsum_W - z_w * sum_x + K * z_w * z_x
y_int    = (s_w * s_x) * int_expr + b

# A1：整数表达式与「浮点反量化后 GEMM」逐位一致（说明没有引入新的近似）
float_expr = (W_q - z_w) @ (x_q - z_x)
check(np.array_equal(int_expr, float_expr.astype(np.int64)),
      "A1 整数表达式与浮点反量化后 GEMM 逐位一致")

# A2：融合路径与 QDQ 参考的数值偏差，应当只有浮点舍入量级
d_fuse  = np.abs(y_ref - y_int).max()
d_quant = np.abs(y_ref - (W @ x + b)).max()
print(f"\nA2 max|y_ref - y_int| = {d_fuse:.3e}")
print(f"   量化误差量级 |y_ref - y_fp| = {d_quant:.3e}")
print(f"   比值 = {d_fuse / d_quant:.3e}  （融合引入的误差相对量化误差可忽略）")
check(d_fuse < 1e-12 and d_fuse / d_quant < 1e-10,
      "A2 融合路径与 QDQ 参考的偏差仅为浮点舍入（<1e-12）")

s_W=0.003330 z_W=12 | s_x=0.021108 z_x=122 | K=256
  [ok] A1 整数表达式与浮点反量化后 GEMM 逐位一致

A2 max|y_ref - y_int| = 8.882e-16
   量化误差量级 |y_ref - y_fp| = 5.334e-02
   比值 = 1.665e-14  （融合引入的误差相对量化误差可忽略）
  [ok] A2 融合路径与 QDQ 参考的偏差仅为浮点舍入（<1e-12）


## B. 三项修正的消融：漏掉任何一项会发生什么

三项修正都不是「可选优化」——它们是把 $$q_W q_x$$ 还原成 $$W x$$ 所**必须**的代数项。
下面逐项拆掉，看误差量级。

In [4]:
y_scale = np.abs(y_ref).max()
variants = [
    ("完整三项",            int_expr),
    ("去掉 z_x·rowsum(W)",  acc - z_w * sum_x + K * z_w * z_x),
    ("去掉 z_W·sum(x)",     acc - z_x * rowsum_W + K * z_w * z_x),
    ("去掉 K·z_W·z_x",      acc - z_x * rowsum_W - z_w * sum_x),
    ("裸整数 GEMM",          acc),
]
print(f"{'变体':<22}{'max err':>14}{'相对误差':>12}{'相对完整版':>14}")
print("-" * 64)
rows = []
for name, expr in variants:
    err = np.abs((s_w * s_x) * expr + b - y_ref).max()
    rel = err / y_scale
    rows.append((name, err, rel))
    print(f"{name:<22}{err:>14.6e}{rel:>12.4f}{rel / rows[0][2]:>14.1f}x")

full_err = rows[0][1]
for name, err, rel in rows[1:]:
    check(err / full_err > 1e8, f"B 漏掉『{name}』后误差放大 {err / full_err:.2e} 倍（量级失效）")
check(rows[-1][2] > 1.0, f"B 裸整数 GEMM 相对误差 {rows[-1][2]:.4f} > 1（输出完全不可用）")

print("\n>>> 三项修正是代数必需项，不是精度微调：漏任何一项，输出量级就崩。")

变体                           max err        相对误差         相对完整版
----------------------------------------------------------------
完整三项                    8.881784e-16      0.0000           1.0x
去掉 z_x·rowsum(W)        3.761249e+01      8.827242347899121362976.0x
去掉 z_W·sum(x)           2.538259e+01      5.957028578255261376840.0x
去掉 K·z_W·z_x            2.634418e+01      6.182729660908823717980.0x
裸整数 GEMM                3.665090e+01      8.601541265245559021824.0x
  [ok] B 漏掉『去掉 z_x·rowsum(W)』后误差放大 4.23e+16 倍（量级失效）
  [ok] B 漏掉『去掉 z_W·sum(x)』后误差放大 2.86e+16 倍（量级失效）
  [ok] B 漏掉『去掉 K·z_W·z_x』后误差放大 2.97e+16 倍（量级失效）
  [ok] B 漏掉『裸整数 GEMM』后误差放大 4.13e+16 倍（量级失效）
  [ok] B 裸整数 GEMM 相对误差 8.6015 > 1（输出完全不可用）

>>> 三项修正是代数必需项，不是精度微调：漏任何一项，输出量级就崩。


## C. 对称量化为什么在工程上更省

权重用对称量化时 $$z_W = 0$$，于是后面两项自动归零：

$$\text{int\_expr} = q_W q_x - z_x\,\textstyle\sum_k q_W[:,k]
                    \underbrace{-\, z_W\sum_k q_x[k]}_{=0}
                    \underbrace{+\, K z_W z_x}_{=0}$$

省掉的是**每输出通道一次乘加**与**一个标量**。这也是「权重量化普遍用对称、激活量化用非对称」
的工程理由：激活有 ReLU 之类的下限，需要 $$z_x$$ 把零点挪到真实分布上；
权重的分布近似零均值对称，强制 $$z_W=0$$ 损失很小却能让 kernel 少算两项。

In [5]:
s_ws, z_ws, _, _ = qparams(W, 8, signed=True, symmetric=True)
W_qs = quantize(W, s_ws, z_ws, wmin, wmax)
print(f"对称权重: z_W = {z_ws}")

term_x = z_x * W_qs.sum(axis=1)
term_w = z_ws * x_q.sum()
term_c = K * z_ws * z_x
print(f"  term_W = z_W·sum(x)   = {term_w}")
print(f"  term_c = K·z_W·z_x    = {term_c}")
print(f"  term_x = z_x·rowsum(W): 非零 {int((term_x != 0).sum())}/{OUT} 项（必须保留）")

check(term_w == 0 and term_c == 0, "C 对称权重下 term_W 与 term_c 自动归零")
check((term_x != 0).sum() > 0, "C term_x 仍然非零（激活非对称时不可省）")

int_s = W_qs @ x_q - term_x - term_w + term_c
y_ref_s = dequantize(W_qs, s_ws, z_ws) @ dequantize(x_q, s_x, z_x) + b
check(np.allclose((s_ws * s_x) * int_s + b, y_ref_s, atol=1e-12),
      "C 对称权重下整数路径与浮点参考一致")

# 代价：对称量化放弃了零点平移，权重量化误差会变大
def snr(ref, got):
    num = np.sum(ref ** 2); den = np.sum((ref - got) ** 2)
    return 10 * np.log10(num / den)
snr_sym = snr(W, dequantize(W_qs, s_ws, z_ws))
snr_asy = snr(W, dequantize(W_q, s_w, z_w))
print(f"\n权重重构 SNR: 对称 {snr_sym:.2f} dB | 非对称 {snr_asy:.2f} dB | 差 {snr_asy - snr_sym:+.2f} dB")
print(f"本次探针的权重是零均值 randn，故差距小；真实模型权重若明显偏置，这个差值会变大。")
check(snr_asy >= snr_sym - 1e-9, "C 非对称量化的权重重构 SNR 不低于对称（付出了精度才换来的）")

对称权重: z_W = 0
  term_W = z_W·sum(x)   = 0
  term_c = K·z_W·z_x    = 0
  term_x = z_x·rowsum(W): 非零 127/128 项（必须保留）
  [ok] C 对称权重下 term_W 与 term_c 自动归零
  [ok] C term_x 仍然非零（激活非对称时不可省）
  [ok] C 对称权重下整数路径与浮点参考一致

权重重构 SNR: 对称 39.45 dB | 非对称 40.31 dB | 差 +0.86 dB
本次探针的权重是零均值 randn，故差距小；真实模型权重若明显偏置，这个差值会变大。
  [ok] C 非对称量化的权重重构 SNR 不低于对称（付出了精度才换来的）


## D. requantize：INT32 累加器怎么回到 INT8

整数 GEMM 的累加器是 INT32，而下一层要的是 INT8。这一步叫 **requantize**：

$$q_y = \mathrm{clip}\Big(\mathrm{round}\big(M \cdot (\mathrm{acc} + \hat b)\big) + z_y,\; q_{\min},\, q_{\max}\Big),
\qquad M = \frac{s_W s_x}{s_y}$$

其中 bias 折进累加器： $$\hat b = \mathrm{round}(b / (s_W s_x))$$（这也是为什么 bias 通常存 INT32）。
关键在 $$M$$：它是个小于 1 的浮点数，必须变成「整数乘子 + 右移」才能在整数单元上算。

按 Jacob et al. 的口径把 $$M$$ 归一化成尾数 $$M_0$$ 与移位量：

$$M = M_0 \cdot 2^{-\mathrm{shift}},\qquad M_0 \in [0.5, 1) \text{ 用 32 位表示}$$

In [6]:
y_for_q = (s_w * s_x) * int_expr + b
s_y, z_y, ymin, ymax = qparams(y_for_q, 8, signed=False)
q_exact = quantize(y_for_q, s_y, z_y, ymin, ymax)

ratio = s_w * s_x / s_y
frac, exp = np.frexp(ratio)
M0    = int(np.round(frac * (1 << 31)))
shift = int(31 - exp)   # np.frexp 返回 np.int32，1<<39 会溢出成 0
acc_b = int_expr + np.round(b / (s_w * s_x)).astype(np.int64)

print(f"M = s_W·s_x/s_y = {ratio:.6e}")
print(f"归一化: M0 = {M0} (int32 正数域内: {M0 < (1 << 31)}), shift = {shift}")

# D1：对称舍入的定点路径与浮点精确码字逐位一致（rshift_sym 定义在 A 组）
q_fx = np.clip(rshift_sym(M0 * acc_b.astype(object), shift) + z_y, ymin, ymax)
mismatch = float((np.asarray(q_fx, dtype=np.int64) != q_exact).mean())
print(f"D1 定点 vs 浮点精确码字 不一致率 = {mismatch * 100:.4f}%")
check(np.array_equal(np.asarray(q_fx, dtype=np.int64), q_exact),
      "D1 对称舍入的定点 requantize 与浮点精确码字逐位一致")

# D2：位宽陷阱 —— M0 是 int32，但 M0*acc 必须走 int64
max_prod = int(np.abs(M0 * acc_b).max())
bits_needed = int(np.ceil(np.log2(max_prod))) + 1
print(f"\nD2 M0 占 32 bit；M0·acc 峰值 = {max_prod:.4e}，需要 {bits_needed} bit")
print(f"   int32 上限 2.147e9 | int64 上限 9.223e18")
check(bits_needed > 32, f"D2 中间乘积需 {bits_needed} bit > 32，必须用 int64 累加（不是实现偷懒）")

# D3：M0 的表示位宽扫描（对称舍入口径）
print(f"\nD3 M0 表示位宽扫描（对称舍入）:")
print(f"{'M0 位宽':>8}{'M0':>14}{'码字不一致率':>14}{'反量化 max err':>18}")
print("-" * 56)
d3 = {}
for nb_ in (8, 12, 16, 20, 24, 31):
    m0 = int(np.round(frac * (1 << nb_)))
    sh = int(nb_ - exp)
    qq = np.array(np.clip(rshift_sym(m0 * acc_b.astype(object), sh) + z_y, ymin, ymax), dtype=np.int64)
    mism = float((qq != q_exact).mean())
    err  = np.abs(dequantize(qq, s_y, z_y) - dequantize(q_exact, s_y, z_y)).max()
    d3[nb_] = (mism, err)
    print(f"{nb_:>8}{m0:>14d}{mism * 100:>13.3f}%{err:>18.3e}")

check(d3[8][0] > 0, f"D3 M0 位宽 8 时不精确（不一致率 {d3[8][0] * 100:.3f}%）")
check(d3[12][0] == 0 and d3[16][0] == 0,
      "D3 M0 位宽 >= 12 时与浮点精确码字逐位一致（本探针上 16 位已足够）")
print("\n   注：本探针 K=256、累加器动态范围有限，故 12 位即够。")
print("   K 更大或 scale 极端时所需位宽会上升；工程上统一取 32 位是保险做法。")


M = s_W·s_x/s_y = 2.293195e-03
归一化: M0 = 1260697305 (int32 正数域内: True), shift = 39
D1 定点 vs 浮点精确码字 不一致率 = 0.0000%
  [ok] D1 对称舍入的定点 requantize 与浮点精确码字逐位一致

D2 M0 占 32 bit；M0·acc 峰值 = 7.6422e+13，需要 48 bit
   int32 上限 2.147e9 | int64 上限 9.223e18
  [ok] D2 中间乘积需 48 bit > 32，必须用 int64 累加（不是实现偷懒）

D3 M0 表示位宽扫描（对称舍入）:
   M0 位宽            M0        码字不一致率       反量化 max err
--------------------------------------------------------
       8           150        6.250%         3.065e-02
      12          2405        0.000%         0.000e+00
      16         38473        0.000%         0.000e+00
      20        615575        0.000%         0.000e+00
      24       9849198        0.000%         0.000e+00
      31    1260697305        0.000%         0.000e+00
  [ok] D3 M0 位宽 8 时不精确（不一致率 6.250%）
  [ok] D3 M0 位宽 >= 12 时与浮点精确码字逐位一致（本探针上 16 位已足够）

   注：本探针 K=256、累加器动态范围有限，故 12 位即够。
   K 更大或 scale 极端时所需位宽会上升；工程上统一取 32 位是保险做法。


### D4. requantize 的舍入口径：三种写法各差在多近的地方

`round` 落在整数单元上有三种常见写法，它们只在**平局（tie）**与**负数**两处不同：

| 写法 | 表达式 | tie 行为 | 负数分支 |
|---|---|---|---|
| 算术右移（截断） | `v >> sh` | 向零/向下取整 | **有偏**（全部向下） |
| 加半再右移 | `(v + 2^{sh-1}) >> sh` | round-half-up | 与正数同口径 |
| 对称加半右移 | `sign(v)·((|v| + 2^{sh-1}) >> sh)` | round-half-away | 与正数同口径 |

关键结论：**「加半个 LSB 再算术右移」已经是无偏的**——因为 `>>` 对负数是 floor，
而 `floor(y + 0.5)` 恰好就是 round-half-up，对正负两侧完全一致。
真正有偏的是**不带加半的裸右移**：它让所有输出码字向下偏半个 LSB。


In [7]:
from fractions import Fraction
def rshift_naive(v, sh):
    """加半个 LSB 再算术右移（工程上最常见的写法，即 A 组的 rshift）。"""
    return (v + (1 << (sh - 1))) >> sh

def rshift_trunc(v, sh):
    """裸算术右移（不带加半）—— 这是真正有偏的写法。"""
    return v >> sh

rng  = np.random.default_rng(1)
# 取值范围必须保证 M0·a 不溢出 int64 —— 否则测的就不是舍入而是溢出（见 D2）
# 安全上界：|a| < 2^63 / M0
a_hi  = int((2 ** 63 - 1) // M0)
accs  = rng.integers(-a_hi, a_hi, size=100000)
print(f"D4 样本范围 |a| < 2^{int(np.log2(a_hi))}（由 M0·a < int64 上限推出）")

# 参考值用 Fraction 精确计算：ratio 是浮点近似（相对误差 ~1e-10），
# 乘以 |a| ~ 1e9 后绝对误差可达 1e-1，会把 0.5 的下界污染成 0.502。
def exact_ref(a):
    return Fraction(M0, 1 << shift) * a

rows = []
for name, fn in [("裸右移 v>>sh", rshift_trunc),
                 ("加半右移 (v+2^{sh-1})>>sh", rshift_naive),
                 ("对称加半右移", rshift_sym_scalar)]:
    d = np.array([float(fn(M0 * int(a), shift) - exact_ref(int(a))) for a in accs])
    rows.append((name, d.mean(), np.abs(d).mean(), np.abs(d).max()))

print(f"{'写法':<28}{'有符号均值':>14}{'无符号均值':>12}{'max|e|':>10}{'判定':>10}")
print("-" * 76)
for name, sm, um, mx in rows:
    verdict = "有偏" if abs(sm) > 0.05 else "无偏"
    print(f"{name:<28}{sm:>+14.5f}{um:>12.5f}{mx:>10.3f}{verdict:>10}")

# D4-1：裸右移确实有偏（约 -0.5）
check(abs(rows[0][1] + 0.5) < 0.05,
      f"D4-1 裸算术右移引入系统性偏置（实测有符号均值 {rows[0][1]:+.5f}）")
# D4-2：加半右移无偏（这是关键更正）
check(abs(rows[1][1]) < 0.05,
      f"D4-2 加半再右移是无偏的（实测有符号均值 {rows[1][1]:+.5f}）")
check(rows[1][3] <= 0.5 + 1e-6,
      f"D4-2 加半右移的 max|e| = {rows[1][3]:.6f} <= 0.5（半个 LSB，理论下界）")
check(abs(rows[2][1]) < 0.05,
      f"D4-3 对称加半右移同样无偏（实测 {rows[2][1]:+.5f}）")

# 偏置是常数：不同 scale 组合下都成立
print("\n不同 scale 组合下有符号均值（裸右移应恒为 -0.5，加半右移应恒为 0）:")
m_trunc, m_half = [], []
for sw_ in (1e-2, 7e-5, 3e-6):
    for sx_ in (1e-2, 2e-3):
        for sy_ in (3e-2, 5e-4):
            r_ = sw_ * sx_ / sy_
            f_, e_ = np.frexp(r_)
            m_ = int(np.round(f_ * (1 << 31))); sh_ = int(31 - e_)
            sub = accs[:4000]
            m_trunc.append(np.mean([float(rshift_trunc(m_ * int(a), sh_)
                                          - Fraction(m_, 1 << sh_) * int(a)) for a in sub]))
            m_half.append(np.mean([float(rshift_naive(m_ * int(a), sh_)
                                         - Fraction(m_, 1 << sh_) * int(a)) for a in sub]))
print(f"  裸右移  : 均值 = {np.mean(m_trunc):+.5f}, 标准差 = {np.std(m_trunc):.5f}")
print(f"  加半右移: 均值 = {np.mean(m_half):+.5f}, 标准差 = {np.std(m_half):.5f}")
check(abs(np.mean(m_trunc) + 0.5) < 0.05 and np.std(m_trunc) < 0.05,
      f"D4-4 裸右移的偏置是常数 -0.5（12 组标准差 {np.std(m_trunc):.5f}）")
check(abs(np.mean(m_half)) < 0.05 and np.std(m_half) < 0.05,
      f"D4-5 加半右移在 12 组 scale 下均无偏（标准差 {np.std(m_half):.5f}）")

print("\n>>> 更正一条常见误传：「加半再右移对负数有偏」是错的。")
print("    floor(y+0.5) 对正负一致地给出 round-half-up。真正有偏的是不带加半的裸右移。")


D4 样本范围 |a| < 2^32（由 M0·a < int64 上限推出）


写法                                   有符号均值       无符号均值    max|e|        判定
----------------------------------------------------------------------------
裸右移 v>>sh                         -0.49933     0.49933     1.000        有偏
加半右移 (v+2^{sh-1})>>sh             -0.00084     0.25013     0.500        无偏
对称加半右移                            -0.00084     0.25013     0.500        无偏
  [ok] D4-1 裸算术右移引入系统性偏置（实测有符号均值 -0.49933）
  [ok] D4-2 加半再右移是无偏的（实测有符号均值 -0.00084）
  [ok] D4-2 加半右移的 max|e| = 0.499998 <= 0.5（半个 LSB，理论下界）
  [ok] D4-3 对称加半右移同样无偏（实测 -0.00084）

不同 scale 组合下有符号均值（裸右移应恒为 -0.5，加半右移应恒为 0）:


  裸右移  : 均值 = -0.50004, 标准差 = 0.00446
  加半右移: 均值 = -0.00156, 标准差 = 0.00583
  [ok] D4-4 裸右移的偏置是常数 -0.5（12 组标准差 0.00446）
  [ok] D4-5 加半右移在 12 组 scale 下均无偏（标准差 0.00583）

>>> 更正一条常见误传：「加半再右移对负数有偏」是错的。
    floor(y+0.5) 对正负一致地给出 round-half-up。真正有偏的是不带加半的裸右移。


## E. 三笔收益账：加速到底来自哪里

一个 $$M \times N \times K$$ 的 GEMM（$$M$$ 是 token 数），算术强度（每搬运 1 字节做多少次运算）是

$$\mathrm{AI} = \frac{2MNK}{NK\, b_W/8 + MK\, b_A/8 + MN\, b_A/8}$$

**M=1（decode 阶段，逐 token 生成）时**它退化成一个漂亮的闭式：

$$\mathrm{AI}\big|_{M=1} = \frac{2NK}{NK\,b_W/8 + K\,b_A/8 + N\,b_A/8}
  \;\xrightarrow{N=K\gg 1}\; \frac{16}{b_W}$$

**激活位宽在这一阶上完全消失了。** 这就是 W4A16 的谜底：它把激活留在 FP16，
却仍然拿到接近 4 倍的收益——因为 decode 阶段搬权重是主要开销，而权重从 16 bit 压到 4 bit。

In [8]:
def ai(b_W, b_A, N, K, M):
    return 2 * M * N * K / (N * K * b_W / 8 + M * K * b_A / 8 + M * N * b_A / 8)

N = 4096
cfgs = [("FP16", 16, 16), ("W8A8", 8, 8), ("W4A16", 4, 16), ("W4A8", 4, 8)]
print(f"{'M':>6}" + "".join(f"{n:>12}" for n, _, _ in cfgs) + "   | W4A16/FP16")
print("-" * 76)
table = {}
for M in (1, 8, 64, 512, 4096):
    vals = [ai(bw, ba, N, N, M) for _, bw, ba in cfgs]
    table[M] = vals
    print(f"{M:>6}" + "".join(f"{v:>12.3f}" for v in vals) + f"   | {vals[2] / vals[0]:>11.3f}x")

# E1: M=1 闭式校验 AI ≈ 16/b_W，与激活位宽无关
print("\nE1 M=1 闭式 16/b_W 校验:")
for name, bw, ba in cfgs:
    got, want = ai(bw, ba, N, N, 1), 16 / bw
    print(f"  {name:<7} AI={got:8.4f}  16/b_W={want:8.4f}  相对偏差 {abs(got - want) / want * 100:.4f}%")
check(all(abs(ai(bw, ba, N, N, 1) - 16 / bw) / (16 / bw) < 0.02 for _, bw, ba in cfgs),
      "E1 M=1 时 AI ≈ 16/b_W（激活位宽在一阶上消失）")

# E2: W4A16 与 W4A8 在 M=1 时几乎相同 —— 激活位宽不影响主收益
w4a16, w4a8 = table[1][2], table[1][3]
print(f"\nE2 M=1: W4A16 AI={w4a16:.4f} vs W4A8 AI={w4a8:.4f}  差 {abs(w4a16 - w4a8) / w4a16 * 100:.4f}%")
check(abs(w4a16 - w4a8) / w4a16 < 0.02,
      "E2 M=1 时 W4A16 与 W4A8 的算术强度几乎相同（收益来自权重位宽）")

# E3：权重位宽与激活位宽的收益方向相反
#   只动激活（W4A16 -> W4A8）的增益，随 M 增大才显现
r_act = []
for M in (1, 8, 64, 512, 4096):
    r_act.append(ai(4, 8, N, N, M) / ai(4, 16, N, N, M))
#   只动权重（W8A8 -> W4A8）的增益，在 M=1 最大，随 M 增大被激活项摊薄
r_wgt = []
for M in (1, 8, 64, 512, 4096):
    r_wgt.append(ai(4, 8, N, N, M) / ai(8, 8, N, N, M))

print("\nE3 权重位宽 vs 激活位宽，收益随 batch 的走向相反:")
print(f"{'M':>6}{'只动激活(W4A16→W4A8)':>24}{'只动权重(W8A8→W4A8)':>24}")
print("-" * 56)
for M, ra, rw in zip((1, 8, 64, 512, 4096), r_act, r_wgt):
    print(f"{M:>6}{ra:>24.4f}{rw:>24.4f}")

check(r_act[0] < 1.01 and r_act[-1] > 1.5,
      f"E3 激活位宽的收益随 batch 增大才显现（M=1 时 {r_act[0]:.3f}x -> M=4096 时 {r_act[-1]:.3f}x）")
check(r_wgt[0] > r_wgt[-1] and r_wgt[0] > 1.9,
      f"E3 权重位宽的收益在 M=1 最大（{r_wgt[0]:.3f}x），随 batch 增大被激活项摊薄到 {r_wgt[-1]:.3f}x")
check(r_act[0] < r_wgt[0],
      f"E3 decode 阶段（M=1）权重位宽的收益（{r_wgt[0]:.3f}x）远大于激活位宽（{r_act[0]:.3f}x）")


     M        FP16        W8A8       W4A16        W4A8   | W4A16/FP16
----------------------------------------------------------------------------
     1       1.000       1.999       3.992       3.996   |       3.994x
     8       7.969      15.938      31.508      31.752   |       3.954x
    64      62.061     124.121     227.556     240.941   |       3.667x
   512     409.600     819.200    1024.000    1365.333   |       2.500x
  4096    1365.333    2730.667    1820.444    3276.800   |       1.333x

E1 M=1 闭式 16/b_W 校验:
  FP16    AI=  0.9995  16/b_W=  1.0000  相对偏差 0.0488%
  W8A8    AI=  1.9990  16/b_W=  2.0000  相对偏差 0.0488%
  W4A16   AI=  3.9922  16/b_W=  4.0000  相对偏差 0.1949%
  W4A8    AI=  3.9961  16/b_W=  4.0000  相对偏差 0.0976%
  [ok] E1 M=1 时 AI ≈ 16/b_W（激活位宽在一阶上消失）

E2 M=1: W4A16 AI=3.9922 vs W4A8 AI=3.9961  差 0.0976%
  [ok] E2 M=1 时 W4A16 与 W4A8 的算术强度几乎相同（收益来自权重位宽）

E3 权重位宽 vs 激活位宽，收益随 batch 的走向相反:
     M        只动激活(W4A16→W4A8)         只动权重(W8A8→W4A8)
---------------------------

### E4. 三笔账分开算：算力、带宽、显存

| 账 | 谁在省 | 什么时候生效 | 上限 |
|---|---|---|---|
| **算力** | 整数单元 vs 浮点单元的峰值比 | 计算受限（大 batch / prefill） | 硬件峰值比，通常 2–4× |
| **带宽** | 权重与激活的字节数 | **访存受限（decode / 小 batch）** | $$16/b_W$$，与激活位宽无关 |
| **显存** | 权重驻留 + KV cache | 常驻开销，与 batch 无关 | 压缩比 |

In [9]:
P = 7e12      # 峰值算力 (FLOP/s)，示意量级
BWm = 3e12    # 峰值带宽 (B/s)，示意量级

print("三笔账（示意常数量级，稳定的是比值与量级差）")
print(f"{'格式':<8}{'算力账':>10}{'带宽账(M=1)':>14}{'显存账':>10}{'实际瓶颈':>14}")
print("-" * 58)
for name, bw, ba in cfgs:
    a = ai(bw, ba, N, N, 1)
    t_flops = 1.0                       # 相对 FP16 的算力账（整数单元峰值比，示意）
    t_bw    = (16 / bw) / (16 / 16)     # 带宽账 = 权重压缩比
    t_mem   = 16 / bw
    bottle  = "带宽" if a < 300 else "算力"
    print(f"{name:<8}{t_flops:>10.2f}{t_bw:>14.2f}{t_mem:>10.2f}{bottle:>14}")

print("\n>>> 读法：decode(M=1) 时 AI 远低于 roofline 拐点，是带宽受限，")
print("    所以 W4A16 的 4x 来自『权重字节数除以 4』，不是来自整数算力。")
print("    整数算力那笔账要到 prefill / 大 batch 才开始兑现。")

check(table[1][2] / table[1][0] > 3.5,
      f"E4 W4A16 在 M=1 时算术强度增益 {table[1][2] / table[1][0]:.3f}x，接近权重压缩比 4x")
check(table[1][0] < 2 and table[4096][0] > 1000,
      f"E4 M=1 时 AI={table[1][0]:.3f}（带宽受限），M=4096 时 AI={table[4096][0]:.1f}（算力受限）")

三笔账（示意常数量级，稳定的是比值与量级差）
格式             算力账      带宽账(M=1)       显存账          实际瓶颈
----------------------------------------------------------
FP16          1.00          1.00      1.00            带宽
W8A8          1.00          2.00      2.00            带宽
W4A16         1.00          4.00      4.00            带宽
W4A8          1.00          4.00      4.00            带宽

>>> 读法：decode(M=1) 时 AI 远低于 roofline 拐点，是带宽受限，
    所以 W4A16 的 4x 来自『权重字节数除以 4』，不是来自整数算力。
    整数算力那笔账要到 prefill / 大 batch 才开始兑现。
  [ok] E4 W4A16 在 M=1 时算术强度增益 3.994x，接近权重压缩比 4x
  [ok] E4 M=1 时 AI=1.000（带宽受限），M=4096 时 AI=1365.3（算力受限）


## F. 汇总

| 组 | 结论 |
|---|---|
| 0 | 本机无量化 engine：QDQ 可导出，整数 kernel 缺席时执行层退回浮点 |
| A | 融合恒等式逐位成立：整数 GEMM + 三项修正 == 浮点反量化后 GEMM |
| B | 三项修正不可省：漏任一项，误差放大 $$10^{8}$$ 倍以上 |
| C | 对称权重让后两项修正归零，代价是权重重构 SNR 的下降 |
| D | requantize 的 $$M_0$$ 是 int32，但中间乘积需 int64；naive 右移引入 −0.5 直流偏置 |
| E | W4A16 的收益在带宽不在算力：M=1 时 AI → $$16/b_W$$，与激活位宽无关 |

In [10]:
print("=" * 60)
print(f"assertions passed: {N_ASSERT}")
print("=" * 60)

assertions passed: 32
